In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "mobilenet_v2",
    "learning_rate": 0.001,
    "epochs": 10,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5],
                             [0.5, 0.5, 0.5])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [ ]:

mobilenet_v2 = models.mobilenet_v2(pretrained=True)
mobilenet_v2 = models.mobilenet_v2(pretrained=True)

for param in mobilenet_v2.parameters():
    param.requires_grad = False

mobilenet_v2.classifier[1] = nn.Linear(mobilenet_v2.last_channel, 84)

gpu = torch.device("cuda")
mobilenet_v2 = mobilenet_v2.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


: 

In [20]:
for param in mobilenet_v2.features[-3:].parameters():
    param.requires_grad = True

In [25]:
for param in mobilenet_v2.features[-6:].parameters():
    param.requires_grad = True

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": mobilenet_v2.classifier[1].parameters(), "lr": 1e-3},
    {"params": mobilenet_v2.features.parameters(), "lr": 1e-4},
    
])
epochs = model_config["epochs"]

In [22]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [26]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(mobilenet_v2, val_dataloader)

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [27]:
train_model(mobilenet_v2, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.1852098238215111, Validation Accuracy: 0.9241208758067435
Epoch: 2, Training Loss: 0.1611988878855206, Validation Accuracy: 0.9267145183666083
Epoch: 3, Training Loss: 0.13883055086624577, Validation Accuracy: 0.9294891127329754
Epoch: 4, Training Loss: 0.12258695778252977, Validation Accuracy: 0.9273780083237831
Epoch: 5, Training Loss: 0.10957830025497264, Validation Accuracy: 0.9287049882381325
Epoch: 6, Training Loss: 0.09700279663552638, Validation Accuracy: 0.929971650883648
Epoch: 7, Training Loss: 0.08760840899339382, Validation Accuracy: 0.9308764099161589
Epoch: 8, Training Loss: 0.08369155366160155, Validation Accuracy: 0.9279811810121238
Epoch: 9, Training Loss: 0.0747508528969654, Validation Accuracy: 0.929066891851137
Epoch: 10, Training Loss: 0.07477385651167442, Validation Accuracy: 0.9282827673562941


In [28]:
accuracy = validate_model(mobilenet_v2, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 92.3638439124035
